# EDA tập phát triển — ERIS Attrition Model API

## 1 — Phạm vi

EDA chỉ sử dụng **development set**. Final test tiếp tục bị khóa, không được đọc, thống kê hoặc dùng để ra quyết định feature. IBM HR Employee Attrition là benchmark công khai, giả lập; dữ liệu không có snapshot date và prediction horizon nên không hỗ trợ diễn giải theo thời gian hoặc quan hệ nhân quả.

In [ ]:
from pathlib import Path
import math
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)

## 2 — Load và validation

Notebook chỉ tìm hai tên development split đã biết; không glob hoặc tham chiếu final test. `source_row`, `EmployeeNumber` và `Attrition` được kiểm tra nhưng không phải model feature.

In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
development_candidates = [repo_root / "data/splits/development_v1.csv", repo_root / "data/processed/development_raw_v1.csv"]
development_paths = [path for path in development_candidates if path.is_file()]
if len(development_paths) != 1:
    raise FileNotFoundError(f"Cần đúng một development split, tìm thấy {len(development_paths)}: {development_paths}")
development_path = development_paths[0]
if any(token in development_path.name.lower() for token in ("final", "locked")):
    raise RuntimeError("Từ chối đọc final test hoặc locked test trong EDA.")
candidate_features = [
    "Age", "BusinessTravel", "Department", "DistanceFromHome", "Education", "EducationField",
    "EnvironmentSatisfaction", "JobInvolvement", "JobLevel", "JobRole", "JobSatisfaction",
    "MonthlyIncome", "NumCompaniesWorked", "OverTime", "PercentSalaryHike", "PerformanceRating",
    "RelationshipSatisfaction", "StockOptionLevel", "TotalWorkingYears", "TrainingTimesLastYear",
    "WorkLifeBalance", "YearsAtCompany", "YearsInCurrentRole", "YearsSinceLastPromotion", "YearsWithCurrManager",
]
non_model_columns = ["source_row", "EmployeeNumber", "Attrition"]
df = pd.read_csv(development_path)
if df.shape != (1176, 28):
    raise RuntimeError(f"Development shape không hợp lệ: {df.shape}; kỳ vọng (1176, 28).")
missing_columns = sorted(set(candidate_features + non_model_columns) - set(df.columns))
if missing_columns:
    raise RuntimeError(f"Thiếu cột bắt buộc: {missing_columns}")
if len(candidate_features) != 25:
    raise RuntimeError(f"Sai số candidate features: {len(candidate_features)}; kỳ vọng 25.")
if set(df["Attrition"].dropna().unique()) == {"No", "Yes"}:
    df["Attrition"] = df["Attrition"].map({"No": 0, "Yes": 1})
target_counts = df["Attrition"].value_counts().sort_index().to_dict()
if target_counts != {0: 986, 1: 190}:
    raise RuntimeError(f"Phân bố Attrition không hợp lệ: {target_counts}; kỳ vọng {{0: 986, 1: 190}}.")
validation_summary = pd.DataFrame({
    "check": ["shape", "candidate_features", "missing_cells", "duplicate_rows", "duplicate_EmployeeNumber", "target_distribution"],
    "result": [str(df.shape), len(candidate_features), int(df.isna().sum().sum()), int(df.duplicated().sum()), int(df["EmployeeNumber"].duplicated().sum()), str(target_counts)],
})
display(validation_summary)
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))
print(f"Development file: {development_path.relative_to(repo_root)}")

## 3 — Phân bố target

In [ ]:
attrition_rate = df["Attrition"].mean()
baseline_accuracy = (df["Attrition"] == 0).mean()
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="Attrition", order=[0, 1], color="#4472C4", ax=ax)
for container in ax.containers:
    ax.bar_label(container)
ax.set(title="Phân bố Attrition trong development set", xlabel="Attrition", ylabel="Số nhân viên")
plt.show()
print(f"Class 0: {target_counts[0]:,} | Class 1: {target_counts[1]:,}")
print(f"Tỷ lệ nghỉ việc: {attrition_rate:.2%}")
print(f"Baseline accuracy nếu luôn đoán class 0: {baseline_accuracy:.2%}")
print("Cảnh báo: dữ liệu mất cân bằng; accuracy đơn lẻ không phản ánh chất lượng model.")

## 4 — Chất lượng dữ liệu

In [ ]:
def compact_feature_summary(frame, features):
    rows = []
    for feature in features:
        series = frame[feature]
        numeric = pd.api.types.is_numeric_dtype(series)
        rows.append({
            "feature": feature, "dtype": str(series.dtype), "missing_count": int(series.isna().sum()),
            "unique_count": int(series.nunique(dropna=True)),
            "min_or_categories": series.min() if numeric else ", ".join(map(str, series.dropna().unique()[:6])),
            "max_or_most_common": series.max() if numeric else str(series.mode(dropna=True).iloc[0]),
        })
    return pd.DataFrame(rows)
quality_table = compact_feature_summary(df, candidate_features)
display(quality_table)

## 5 — Phân bố feature

In [ ]:
numeric_features = [f for f in candidate_features if pd.api.types.is_numeric_dtype(df[f])]
categorical_features = [f for f in candidate_features if f not in numeric_features]
ncols, nrows = 4, math.ceil(len(numeric_features) / 4)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows))
for ax, feature in zip(axes.flat, numeric_features):
    sns.histplot(df[feature], bins=20, ax=ax, color="#4472C4")
    ax.set(title=feature, xlabel=feature, ylabel="Số quan sát")
for ax in axes.flat[len(numeric_features):]: ax.set_visible(False)
fig.suptitle("Phân bố numeric features", y=1.01, fontsize=15)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feature in zip(axes.flat, categorical_features):
    sns.countplot(data=df, y=feature, order=df[feature].value_counts().index, color="#70AD47", ax=ax)
    ax.set(title=feature, xlabel="Số quan sát", ylabel=feature)
for ax in axes.flat[len(categorical_features):]: ax.set_visible(False)
fig.suptitle("Phân bố categorical features", y=1.01, fontsize=15)
plt.tight_layout(); plt.show()
boxplot_features = ["Age", "MonthlyIncome", "TotalWorkingYears", "YearsAtCompany", "YearsInCurrentRole", "YearsWithCurrManager"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.flat, boxplot_features):
    sns.boxplot(data=df, x="Attrition", y=feature, color="#A5A5A5", ax=ax)
    ax.set(title=f"{feature} theo Attrition", xlabel="Attrition", ylabel=feature)
plt.tight_layout(); plt.show()

## 6 — Feature và Attrition

Tỷ lệ luôn đi kèm sample size (`n`). Nhóm dưới 30 quan sát được gắn cờ và không dùng để đưa ra kết luận mạnh.

In [ ]:
categorical_rates = []
for feature in categorical_features:
    rates = df.groupby(feature, dropna=False)["Attrition"].agg(attrition_rate="mean", n="size").reset_index()
    rates.insert(0, "feature", feature)
    rates = rates.rename(columns={feature: "category"})
    rates["small_sample_warning"] = rates["n"] < 30
    categorical_rates.append(rates)
categorical_rate_table = pd.concat(categorical_rates, ignore_index=True)
categorical_rate_table["attrition_rate"] = categorical_rate_table["attrition_rate"].round(3)
display(categorical_rate_table.sort_values(["feature", "attrition_rate"], ascending=[True, False]))
numeric_medians = df.groupby("Attrition")[numeric_features].median().T
numeric_medians.columns = ["median_class_0", "median_class_1"]
display(numeric_medians)
display(categorical_rate_table.sort_values(["attrition_rate", "n"], ascending=[False, False]).head(10))

## 7 — Correlation và rà soát feature

Spearman được dùng để rà soát quan hệ đơn điệu giữa numeric/ordinal features. Correlation cao chỉ là tín hiệu cần kiểm tra; không phải lý do tự động loại feature.

In [ ]:
spearman = df[numeric_features].corr(method="spearman")
mask = pd.DataFrame(False, index=spearman.index, columns=spearman.columns)
for i in range(len(mask)): mask.iloc[i, i:] = True
plt.figure(figsize=(14, 11))
sns.heatmap(spearman, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Spearman correlation — numeric/ordinal features")
plt.tight_layout(); plt.show()
high_corr_pairs = []
for i, left in enumerate(numeric_features):
    for right in numeric_features[i + 1:]:
        value = spearman.loc[left, right]
        if abs(value) >= 0.80:
            high_corr_pairs.append({"feature_1": left, "feature_2": right, "spearman": round(value, 3)})
high_corr_table = pd.DataFrame(high_corr_pairs)
display(high_corr_table if not high_corr_table.empty else pd.DataFrame({"note": ["Không có cặp đạt ngưỡng."]}))
review_notes = {
    "Age": ("review", "Cần đánh giá fairness và hiệu năng theo nhóm tuổi trước khi sử dụng."),
    "JobLevel": ("review", "Tương quan Spearman cao với MonthlyIncome; giữ để so sánh trong CV."),
    "MonthlyIncome": ("review", "Tương quan Spearman cao với JobLevel; chưa đủ bằng chứng để loại."),
    "PerformanceRating": ("review", "Low variance; cần kiểm tra giá trị gia tăng trong cross-validation."),
    "TotalWorkingYears": ("review", "Thuộc nhóm tenure; rà soát tính trùng lặp và tính sẵn có khi dự đoán."),
    "YearsAtCompany": ("review", "Tương quan cao với YearsInCurrentRole và YearsWithCurrManager."),
    "YearsInCurrentRole": ("review", "Tương quan cao với YearsAtCompany; chưa tự động loại."),
    "YearsSinceLastPromotion": ("review", "Thuộc nhóm tenure; rà soát cùng các biến thâm niên."),
    "YearsWithCurrManager": ("review", "Tương quan cao với YearsAtCompany; chưa tự động loại."),
}
ordinal_features = {"Education", "EnvironmentSatisfaction", "JobInvolvement", "JobLevel", "JobSatisfaction", "PerformanceRating", "RelationshipSatisfaction", "StockOptionLevel", "WorkLifeBalance"}
feature_review_rows = []
for feature in candidate_features:
    feature_type = "categorical" if feature in categorical_features else ("ordinal" if feature in ordinal_features else "numeric")
    decision, reason = review_notes.get(feature, ("retain", "Không có vấn đề rõ ràng trong EDA development; xác nhận lại bằng CV."))
    preprocessing = "OneHotEncoder(handle_unknown='ignore')" if feature_type == "categorical" else ("Ordinal passthrough hoặc scale theo pipeline" if feature_type == "ordinal" else "Imputer + StandardScaler trong pipeline")
    feature_review_rows.append({"feature": feature, "type": feature_type, "decision": decision, "reason": reason, "preprocessing": preprocessing})
feature_review = pd.DataFrame(feature_review_rows)
assert feature_review.shape[0] == 25
assert set(feature_review["decision"]) <= {"retain", "review", "drop_proposed"}
display(feature_review)

## 8 — Kết luận

In [ ]:
decision_counts = feature_review["decision"].value_counts().reindex(["retain", "review", "drop_proposed"], fill_value=0)
attention_features = feature_review.loc[feature_review["decision"] == "review", "feature"].tolist()
print(f"Retain: {decision_counts['retain']} | Review: {decision_counts['review']} | Drop proposed: {decision_counts['drop_proposed']}")
print("Feature cần chú ý:", ", ".join(attention_features))
print(f"Class imbalance: class 1 chiếm {attrition_rate:.2%}; ưu tiên metric phù hợp thay vì accuracy đơn lẻ.")
print("Giới hạn: benchmark giả lập, không có snapshot date/prediction horizon, không hỗ trợ kết luận nhân quả.")
print("Bước tiếp theo: ColumnTransformer → DummyClassifier → Logistic Regression → Stratified 5-fold cross-validation")
print("Notebook này không huấn luyện hoặc đánh giá model.")